> **실행 안내**: 이 노트북은 현재 폴더의 `train.csv`, `val.csv`, `test.csv`를 읽는다. 제공된 `inputs/` 폴더에 들어 있는 세 CSV를 노트북과 같은 위치로 복사하고 위에서 아래로 실행하자. Neural Network sweep은 최대 600 epoch이므로 시간이 걸릴 수 있다. 이미 학습한 결과는 제출용 보고서와 `results/` 폴더에 보존되어 있다.

# Homework 2 · Problem 3(a)–(d)
## 처음 보는 12살 학생부터 수학·머신러닝 전문가까지: RKS와 MLP 완전 학습 노트

**학습 목표**: 먼저 2차원 위치에서 높이를 예측한다는 직관을 이해한 뒤, 회귀·MSE·행렬·사인/코사인·랜덤 특징·Ridge 정규방정식·ReLU·역전파·Adam·Early Stopping까지 이해한다. 마지막에는 **교수님이 제공한 진짜 `train.csv`, `val.csv`, `test.csv`**로 3(a)~(d)를 처음부터 끝까지 실행한다.

### 사용한 실제 파일

| 데이터 | 행 수 | 입력 | 정답 |
|---|---:|---|---|
| Train | 600 | `x1`, `x2` | `y` 있음 |
| Validation | 200 | `x1`, `x2` | `y` 있음 |
| Test | 200 | `x1`, `x2` | **`y` 없음** |

**읽는 방법**: `설명 → 바로 아래 코드 셀 → 실제 결과 확인` 순서로 진행한다. 모든 실행문에는 한국어로 의미를 설명한 주석을 달았다. 코드는 독립적인 조각이 아니라 **위에서 아래로 실행하는 완전한 하나의 프로그램**이다.

### 준비

1. 이 `.ipynb`와 `train.csv`, `val.csv`, `test.csv`를 같은 폴더에 둔다. (저장소에서는 `data/` 폴더의 세 파일을 이 폴더로 복사한다.)
2. 설치: `python -m pip install numpy pandas matplotlib torch jupyter`
3. 그 폴더에서 Jupyter를 실행해 **Kernel → Restart & Run All**을 누른다.
4. 모든 그림과 실험표는 `problem3_outputs/`, 두 개의 제출용 예측 파일은 현재 폴더에 생성된다.

---

# Part A. 아무것도 모르는 친구를 위한 출발점

## 1. 점 세 개 `(x1, x2, y)`는 무엇인가?

미니어처 놀이공원 지도를 생각해 보자. 가로 방향 위치가 `x1`, 세로 방향 위치가 `x2`, 그 지점의 땅 높이가 `y`라고 생각하면 된다. 이 과제에서 `x1,x2`는 실제 지도의 좌표가 아니지만 **두 입력으로 하나의 숫자를 예측한다**는 구조는 완전히 같다.

| x1 | x2 | y | 의미 |
|---:|---:|---:|---|
| 0 | 0 | 1 | 가운데의 높이 1 |
| 1 | 0 | 3 | 오른쪽으로 가니 높이 3 |
| 0 | 1 | 2 | 위로 가니 높이 2 |

학습 목표는 `어디인지 알려주면 높이를 맞히는 함수 f(x1,x2)`를 얻는 것이다.

### 데이터 행과 입력 열

- **행(row)** 한 개 = 한 관측치. 지도 위 한 점이다.
- **열(column)** `x1`, `x2` = 입력 2개, `y` = 맞혀야 하는 정답 1개다.
- **모델** = 입력을 보고 예측값 `ŷ`를 계산하는 규칙이다.

`ŷ`는 '와이 햇(y-hat)'이라고 읽으며 **정답이 아닌 모델의 추정값**이라는 뜻이다.

## 2. '잘 맞혔다'는 것을 어떻게 숫자로 정할까? — MSE

정답이 `[1, 3, 5]`이고 모델 예측이 `[1, 5, 4]`라면, 각각의 오차 `(예측-정답)`은 `[0, 2, -1]`이다.

**오차를 그대로 더하면** 큰 플러스와 마이너스가 상쇄된다. 그래서 오차를 제곱해 `[0, 4, 1]`을 만들고, 평균을 낸다.

\[
\mathrm{MSE}=\frac{0+4+1}{3}=\frac53\approx1.667.
\]

일반적으로 관측치가 `n`개이면,

\[
\boxed{\mathrm{MSE}=\frac1n\sum_{i=1}^{n}(y_i-\hat y_i)^2}
\]

으로 쓴다. MSE가 작을수록 예측을 잘한 것이지만, **오차가 큰 한 점에 더 큰 벌점**을 준다.

## 3. 세 갈래의 데이터는 왜 필요할까?

**Train(연습 문제)**은 모델의 계수를 학습할 때만 사용한다. **Validation(모의고사)**은 여러 모델/하이퍼파라미터 중 좋은 것을 선택할 때 사용한다. **Test(최종시험)**는 선택한 모델로 예측값을 만드는 데 사용한다. 이 과제의 test는 답지 `y`가 아예 없으므로 테스트 MSE를 계산할 수 없다.

'Validation을 보고 선택했다'는 것은 이미 Validation 성적에 맞게 여러 후보를 시험해 본 것이다. 따라서 검증 MSE는 모델 선택을 위한 성능 추정치지, 아직 모르는 최종 시험 점수는 아니다.

---

# Part B. 수학 0에서 RKS로

## 4. 왜 직선 하나로는 부족할까?

아주 단순한 예측 함수는

\[
\hat y=b_0+b_1x_1+b_2x_2
\]

이다. `b0`은 기본 높이, `b1`은 오른쪽으로 한 칸 움직일 때의 변화, `b2`는 위로 한 칸 움직일 때의 변화다. 이 함수의 그래프는 **평평한 경사면**이다. 그러나 실제 지형은 산, 골짜기, 물결처럼 굽어 있다. 이때 사용하는 아이디어가 **Feature Transformation(특징 변환)**이다.

## 5. 물결 하나를 만드는 `sin`과 `cos`

사인·코사인은 위아래로 반복되는 물결 함수다. 각도 `t`를 넣으면 `sin(t)`와 `cos(t)`는 둘 다 `-1`에서 `1` 사이의 값을 돌려준다. 여기서는 **라디안(radian)**을 사용한다. 약 `2π`만큼 움직이면 한 주기가 끝난다.

하나의 수를 입력으로 받는 물결을 2차원 위치에도 적용할 수 있다:

\[
\phi_j^{\sin}(x)=\sin(w_j^\top x+b_j),\quad
\phi_j^{\cos}(x)=\cos(w_j^\top x+b_j).
\]

모든 기호를 뜯어보면,

- `x=[x1,x2]`: 지도 위 입력 위치.
- `w_j=[w_{j1},w_{j2}]`: **물결이 어떤 방향으로 얼마나 빠르게 바뀌는지** 정하는 숫자 두 개.
- `w_j^T x = w_{j1} x1+w_{j2} x2`: 두 값을 곱해 더하는 내적(dot product).
- `b_j`: 물결을 좌우로 움직이게 하는 **위상(phase)**.
- `φ`: 새로 만들어진 숫자, 즉 **특징(feature)**.

### 숫자로 직접 계산

`x=(1,2), w=(0.5,-1), b=0.2`라고 해보자.

\[
w^\top x+b=(0.5)(1)+(-1)(2)+0.2=-1.3
\]

따라서 `sin(-1.3) ≈ -0.9636`, `cos(-1.3) ≈ 0.2675`라는 **두 개의 특징**이 생긴다.

## 6. 이게 왜 Random Kitchen Sinks일까?

우리는 특징을 만드는 `w`, `b`를 직접 찾지 않는다. 먼저 정해진 확률분포에서 **무작위로 뽑은 뒤 그 값을 고정**한다.

\[
\boxed{w_j\sim\mathcal N(0,I_2/\sigma^2),\quad b_j\sim U[0,2\pi]}
\]

- `𝒩`은 정규분포. `w`의 각 좌표는 평균 0, **표준편차 `1/σ`**인 정규분포에서 나온다.
- `U[0,2π]`는 0과 `2π` 사이 어디든 같은 가능성으로 뽑는 균등분포다.
- **`σ`가 작으면** `w`가 더 큰 숫자가 되기 쉬워 물결의 변화가 빨라진다. **`σ`가 크면** 천천히 변하는 물결이 만들어지기 쉽다.

하나의 `w,b`마다 `sin`과 `cos` **두 숫자**가 나오므로, `h`쌍이면 전체 특징은 `D=2h`개다. 예: `D=20`이면 서로 다른 10개의 물결에 대한 sin/cos 쌍이다.

\[
\phi(x)=[\sin(w_1^\top x+b_1),\cos(w_1^\top x+b_1),\ldots,\sin(w_h^\top x+b_h),\cos(w_h^\top x+b_h)].
\]

## 7. 랜덤 물결로 만든 선형회귀는 왜 비선형 예측이 될까?

RKS가 만드는 최종 함수는

\[
\hat y(x)=\beta_0+\sum_{k=1}^{D}\beta_k\phi_k(x).
\]

`β`는 학습으로 찾지만 `φ`의 모양은 랜덤으로 정해지고 이후 바뀌지 않는다. `φ`가 사인/코사인이므로, `β`들의 단순한 덧셈조차 원래 좌표 `x1,x2`에 대해서는 **굽이치는 비선형 함수**가 된다.

- **고정되는 것**: 랜덤 주파수 `W`, 랜덤 위상 `b`.
- **학습되는 것**: 위 특징들을 얼마나 섞을지 정하는 `β`.

## 8. 왜 또 Ridge가 필요한가?

특징을 `D=320`개 만들었다면 선형회귀 계수도 절편까지 **321개** 생긴다. 점 600개를 외워 버릴 수 있으므로 큰 계수에 벌점을 준다.

\[
J(\beta)=\underbrace{\|A\beta-y\|_2^2}_{\text{예측오차 제곱의 합}}+
\lambda\underbrace{\beta^\top P\beta}_{\text{절편 제외 계수 제곱합}},
\quad P=\operatorname{diag}(0,1,\ldots,1).
\]

여기서 `A=[1, φ(x)]`는 데이터가 `n`개일 때 `(n,D+1)` 크기의 행렬이다. 첫 열의 `1`은 절편 `β0`를 위한 자리다. 계수 `β`는 길이 `D+1` 벡터다.

### 왜 정규방정식이 되는가?

벡터 `r=Aβ−y`는 각 데이터의 예측오차다. `rᵀr`를 미분하면 `2Aᵀ(Aβ−y)`, Ridge 벌점을 미분하면 `2λPβ`가 나온다.

\[
\nabla J=2A^\top(A\beta-y)+2\lambda P\beta=0
\]

이를 전개·정리하면,

\[
2A^\top A\beta-2A^\top y+2\lambda P\beta=0
\implies \boxed{(A^\top A+\lambda P)\beta=A^\top y}.
\]

코드의 `A.T @ A`, `penalty`, `np.linalg.solve(left,right)`가 **정확히 이 수학 공식**이다. **중요**: 여기의 손실 첫 항은 MSE가 아니라 *제곱오차의 합(SSE)*이다. 평가할 때의 MSE는 그 값을 `n`으로 나눈 것이다. 벌점의 상대적 세기는 이 정의에 따라 달라진다.

### 전문가용 연결 — Gaussian RBF 커널

사인/코사인 덧셈공식을 사용하면 한 쌍의 내적은

\[
\sin(a)\sin(c)+\cos(a)\cos(c)=\cos(a-c).
\]

따라서 \(h\)쌍의 특징에 대해 \(h^{-1}\phi(x)^\top\phi(x')\)는 큰 `h`에서

\[
\mathbb E_w[\cos(w^\top(x-x'))]=
\exp\!\left(-\frac{\|x-x'\|^2}{2\sigma^2}\right)
\]

에 가까워진다. 이 값이 바로 Gaussian RBF 커널이다. **주의**: 과제 구현의 `φ`는 문제에서 지정한 *비정규화* sin/cos 특성이며, RBF 근사 비교에서만 `1/h`를 곱한다. 따라서 코드의 Ridge `λ`는 그 스케일에 맞춰 해석해야 한다.

---

# Part C. 수학 0에서 MLP로

## 9. 뉴런 하나는 무엇인가?

뉴런 하나는 입력 두 개를 받아

\[
z=w_1x_1+w_2x_2+b
\]

라는 가중합을 만든 뒤 `ReLU(z)=max(0,z)`를 통과시킨다. 음수는 0으로 만들고 양수는 그대로 둔다. 이렇게 하면 직선만으로는 만들 수 없었던 **여러 구간의 굴곡**을 표현할 수 있다.

은닉 노드가 `H`개인 **정확히 한 개의 은닉층** 네트워크는

\[
\boxed{\hat y(x)=c+\sum_{j=1}^{H}v_j\operatorname{ReLU}(w_j^\top x+b_j)}
\]

로 쓸 수 있다. 이것은 입력 2개 → 은닉 노드 H개 → 출력 1개로 연결된다. 여기서 RKS와 달리 `w_j`, `b_j`, `v_j`, `c`를 **모두 학습**한다.

## 10. 학습은 어떻게 일어날까? — 작은 숫자 예제

너무 단순한 모델 `ŷ=w·x`에서 `x=2`, 진짜 `y=5`, 초기 `w=1`로 시작하자.

- 처음 예측은 `ŷ=1·2=2`.
- 제곱오차는 `(2−5)²=9`.
- 미분은 `d(w x−y)²/dw = 2(wx−y)x = 2(2−5)·2=−12`.
- 학습률이 `η=0.1`이면 새로운 계수는 `w_new = 1−0.1(−12)=2.2`.
- 다음 예측은 `ŷ=2.2·2=4.4`; 오차제곱은 `(4.4−5)²=0.36`.

즉, **미분으로 어느 방향으로 고쳐야 오차가 줄어드는지 알아내고**, 조금씩 업데이트하는 것이다. 이것이 Gradient Descent의 핵심이다.

### MLP에서는 왜 `loss.backward()`가 가능한가?

출력은 Linear → ReLU → Linear 등 여러 계산을 거친다. 연쇄법칙(chain rule)을 적용하면 `예측을 바꾸는 최종층 → 중간층 → 첫 층`의 기울기를 순차적으로 계산할 수 있다. PyTorch의 자동미분(Autograd)이 그 과정을 수행한다.

`optimizer.step()`은 계산한 기울기를 사용해 실제 가중치 값을 변경한다. 여기서는 단순 SGD 대신 **Adam**을 사용한다. Adam은 기울기의 이동평균과 제곱 이동평균을 이용해 파라미터마다 업데이트 크기를 조정한다.

## 11. Epoch, Batch, Early Stopping이란?

- **Epoch** = 전체 학습 데이터 600행을 한 번 다 보는 과정.
- **Batch** = 한 번에 읽는 작은 묶음. `batch_size=64`이면 600개를 약 10번으로 나누어 업데이트한다.
- **Learning rate** = 업데이트 크기를 조절하는 숫자.
- **Validation MSE** = *새로운 데이터에서* 얼마나 잘하는지 관찰하는 점수.
- **Early Stopping** = Validation이 연속적으로 개선되지 않으면 학습을 종료하고 가장 좋았던 Epoch의 가중치로 되돌리는 방법.

검증에 쓰이는 행들은 **가중치 미분/업데이트**에 직접 사용하지 않는다. 검증값은 모델 선택과 조기종료에만 사용한다. 이 차이가 데이터 누수를 막는 중요한 장치다.

## 12. RKS와 MLP를 한 문장씩 비교

| 구분 | RKS + Ridge | 1-hidden-layer MLP |
|---|---|---|
| 특징 함수 | 무작위로 뽑고 **고정** | 데이터에 맞게 **학습** |
| 출력을 합치는 계수 | Ridge 연립방정식으로 학습 | 역전파 + Adam으로 학습 |
| 예측 형태 | sin/cos의 부드러운 조합 | ReLU로 만든 조각별 선형 조합 |
| 하이퍼파라미터 sweep | 총 특징수 `D` | 은닉층 노드수 `H` |
| 과적합 대응 | Ridge L2 벌점 | 검증 기반 Early Stopping |

**훈련 오차가 낮다는 이유만으로 좋은 모델이라고 결론 내리지 않는다.** 최종 선택 기준은 Validation MSE이다.

---

# Part D. 실제 제출용 전체 Python 코드 — 단계별 한 줄 해설

아래 모든 코드 셀을 **위에서 아래로** 실행한다. 각 실행문에는 한국어로 왜 필요한지 설명하는 주석이 붙어 있다. 코드에서 수학 기호가 어떻게 구현되는지 위 개념 설명과 대조하며 읽자.

### 보너스: MSE와 사인파를 숫자로 직접 확인해 보기

이 셀은 제출 코드와 별개인 짧은 연습이며 이후 변수와 충돌하지 않는다.

In [ ]:
# 12살 친구를 위한 숫자 연습: 오차를 제곱해서 평균 내기
small_true = np.array([1, 3, 5]) if "np" in globals() else [1, 3, 5]  # 아직 NumPy를 import하지 않았으면 일반 리스트를 사용한다.
small_pred = [1, 5, 4]  # 모델이 예측했다고 가정한 값이다.
squared_errors = [(a-b)**2 for a, b in zip(small_true, small_pred)]  # 오차를 각각 제곱한다.
print("제곱오차:", squared_errors, "MSE:", sum(squared_errors)/len(squared_errors))  # MSE=5/3을 확인한다.

### 안내 · 완전한 Python 파일과 동일한 코드

아래부터는 `.py` 파일의 코드 **전체**를 순서대로 사용한다. 각 셀에 붙은 한국어 설명을 읽고 실행하자.

In [ ]:
# -*- coding: utf-8 -*-
"""Homework 2 · Problem 3(a)–(d): RKS + Ridge vs one-hidden-layer MLP.

실행 방법 (파일이 같은 폴더에 있을 때): python problem3_rks_mlp.py  (homework_2 폴더에서는 python run_all.py --problem 3)
필요한 파일: train.csv, val.csv, test.csv
의존 패키지: numpy, pandas, matplotlib, torch
모든 실행문 옆에는 '이 줄이 무엇을 하는가'에 대한 한국어 설명을 적었다.
중요: test.csv에는 정답 y가 없기 때문에 test MSE를 계산하지 않는다.
"""

### 코드 0. `import`는 무엇인가?

새 학교에 입학하기 전 연필과 교과서를 준비하는 것과 같다. `numpy`는 행렬 계산기, `pandas`는 표 계산기, `matplotlib`은 그림 도구, `torch`와 `nn`은 신경망 건축 도구, `DataLoader`는 데이터를 작은 묶음으로 나눠 공급하는 도구다. `matplotlib.use("Agg")`는 화면이 없는 서버에서도 PNG를 저장하게 해 준다. `from torch import nn`이 없으면 `nn.Module`에 밑줄(미정의 오류)이 생긴다.


In [ ]:
# %% 0. 라이브러리: 먼저 필요한 모든 이름을 정의한다.
import copy  # 학습 중 가장 좋은 신경망 가중치를 안전하게 복사한다.
from pathlib import Path  # CSV 및 결과 파일의 경로를 OS에 독립적으로 다룬다.
import numpy as np  # 배열, 행렬 곱, 난수, 선형방정식 해법을 제공한다.
import pandas as pd  # CSV 파일을 읽고 실험 결과를 표로 저장한다.
import matplotlib  # 그림을 파일로 저장하는 Matplotlib의 기본 패키지다.
matplotlib.use("Agg")  # 화면이 없는 환경에서도 PNG 이미지를 저장하도록 설정한다.
import matplotlib.pyplot as plt  # 산점도, 손실 곡선, 등고면을 그린다.
import torch  # 신경망을 계산하고 자동미분하는 PyTorch다.
from torch import nn  # nn.Module, nn.Linear, nn.ReLU, nn.MSELoss 등을 사용한다.
from torch.utils.data import DataLoader, TensorDataset  # 훈련 데이터를 미니배치로 공급한다.

### 코드 1. `SEED`, `DEVICE`, `D`, `λ`는 어디서 나올까?

이제부터 등장하는 **모든 상수는 이 셀에서 처음 정의**된다. `SEED=42`는 실험 재현성, `DEVICE`는 GPU가 있으면 CUDA, 없으면 CPU를 의미한다. `D=20`은 특징 20개=sin/cos 10쌍이며, `RKS_SIGMA=1.0`은 물결의 길이, `RKS_LAMBDA=0.01`은 계수 벌점의 강도를 뜻한다. MLP는 `H∈{16,32,64,128}`, 학습률 `0.002`, 최대 600 Epoch, Early Stopping 45 Epoch를 사용한다. **이 설정값들은 이후 코드 어디에서도 미정의로 사용되지 않는다.**


In [ ]:
# %% 1. 실험의 고정 설정: 이 변수를 바꾸면 전체 실험에 영향을 준다.
SEED = 42  # 같은 프로그램을 다시 실행할 때 재현성을 높이기 위한 숫자다.
np.random.seed(SEED)  # NumPy의 전역 난수 생성 상태를 고정한다.
torch.manual_seed(SEED)  # PyTorch의 가중치 초기화 등 난수 생성 상태를 고정한다.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # CUDA GPU가 있으면 사용하고, 없으면 CPU를 사용한다.
DATA_DIR = Path(".")  # 현재 작업 폴더에서 train.csv, val.csv, test.csv를 찾는다.
OUTPUT_DIR = Path("problem3_outputs")  # 그림과 실험표를 저장할 폴더 이름이다.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # 결과 폴더가 없으면 생성하고, 이미 있으면 넘어간다.
RKS_D_VALUES = [20, 40, 80, 160, 320]  # RKS의 총 sin/cos 특성 개수 D=2h 후보를 정한다.
RKS_SIGMA = 1.0  # 랜덤 주파수의 표준편차를 1/sigma로 지정하는 스케일이다.
RKS_LAMBDA = 0.01  # RKS의 선형회귀 계수에 추가할 Ridge L2 벌점 강도다.
MLP_WIDTH_VALUES = [16, 32, 64, 128]  # MLP의 은닉층 노드 수 후보를 정한다.
MLP_LR = 0.002  # Adam이 가중치를 업데이트할 때 사용하는 학습률이다.
MLP_BATCH_SIZE = 64  # 한 번의 가중치 업데이트에 사용할 데이터 개수다.
MLP_MAX_EPOCHS = 600  # MLP가 전체 학습 데이터를 최대 몇 번 볼지 정한다.
MLP_PATIENCE = 45  # 검증 MSE 개선이 없는 Epoch가 이만큼 쌓이면 멈춘다.
print(f"계산 장치: {DEVICE}")  # 실제 선택된 CPU/GPU를 터미널에서 확인한다.

### 코드 2. CSV를 불러오면 메모리에 어떤 모양으로 저장될까?

`train_df`는 `pandas.DataFrame`이고, `X_train`은 숫자만 들어 있는 NumPy 행렬이다. 실제 모양은 `X_train.shape=(600,2)`, `y_train.shape=(600,)`이다. `X_val=(200,2)`, `y_val=(200,)`, `X_test=(200,2)`도 동일하다. `assert`는 프로그램의 가정이 틀리면 **조용히 잘못된 답을 내지 않고 바로 멈추게** 하는 안전장치다. **test에는 y가 없다.**


In [ ]:
# %% 2. 주어진 CSV 세 개를 읽고, 모양과 내용에 문제가 없는지 확인한다.
train_df = pd.read_csv(DATA_DIR / "train.csv")  # 정답 y가 있는 훈련 데이터 600행을 읽는다.
val_df = pd.read_csv(DATA_DIR / "val.csv")  # 정답 y가 있는 검증 데이터 200행을 읽는다.
test_df = pd.read_csv(DATA_DIR / "test.csv")  # 정답 y가 없는 테스트 데이터 200행을 읽는다.
assert list(train_df.columns) == ["x1", "x2", "y"], "train.csv 열은 x1,x2,y여야 해요"  # 훈련 열 이름과 순서를 검증한다.
assert list(val_df.columns) == ["x1", "x2", "y"], "val.csv 열은 x1,x2,y여야 해요"  # 검증 열 이름과 순서를 검증한다.
assert list(test_df.columns) == ["x1", "x2"], "test.csv 열은 x1,x2여야 해요"  # 테스트에는 y열이 없어야 한다.
X_train = train_df[["x1", "x2"]].to_numpy(dtype=np.float64)  # 훈련의 두 입력 열을 (n_train,2) 배열로 만든다.
y_train = train_df["y"].to_numpy(dtype=np.float64)  # 훈련의 실제 답을 길이 n_train 벡터로 만든다.
X_val = val_df[["x1", "x2"]].to_numpy(dtype=np.float64)  # 검증의 두 입력 열을 (n_val,2) 배열로 만든다.
y_val = val_df["y"].to_numpy(dtype=np.float64)  # 검증의 실제 답을 길이 n_val 벡터로 만든다.
X_test = test_df[["x1", "x2"]].to_numpy(dtype=np.float64)  # 테스트에서는 입력 x1,x2만 사용한다.
for array in (X_train, y_train, X_val, y_val, X_test):  # 모든 데이터 배열을 차례로 점검한다.
    assert np.isfinite(array).all(), "데이터에 NaN/무한대가 있어요"  # 계산 불가능한 숫자가 있는지 검사한다.
assert X_train.ndim == 2 and X_train.shape[1] == 2  # 훈련 데이터의 입력 차원이 정확히 2인지 확인한다.
assert X_val.ndim == 2 and X_val.shape[1] == 2  # 검증 데이터의 입력 차원이 정확히 2인지 확인한다.
assert X_test.ndim == 2 and X_test.shape[1] == 2  # 테스트 데이터의 입력 차원이 정확히 2인지 확인한다.
print(f"데이터: train={len(X_train)} / val={len(X_val)} / test={len(X_test)}")  # 각 분할의 관측치 개수를 보여 준다.

### 코드 3. MSE 공식과 코드 하나하나 대응

수식 `MSE=(1/n)Σ(y_i−ŷ_i)²`에서 `y_true - y_pred`는 오차의 벡터, `** 2`는 각 원소 제곱, `np.mean(...)`은 평균이다. `.reshape(-1)`은 `(n,1)`과 `(n,)`의 형태 차이 때문에 행렬이 엉뚱하게 늘어나는 *브로드캐스팅 오류*를 예방한다.


In [ ]:
# %% 3. 수학의 MSE를 코드로 직접 정의한다.
def mse(y_true, y_pred):  # 실제값 배열과 예측값 배열을 받아 MSE 하나를 계산한다.
    y_true = np.asarray(y_true).reshape(-1)  # 실제값을 모양 (n,)의 1차원 배열로 바꾼다.
    y_pred = np.asarray(y_pred).reshape(-1)  # 예측값도 (n,) 모양으로 통일한다.
    assert y_true.shape == y_pred.shape  # 같은 행끼리 비교할 수 있도록 길이를 확인한다.
    return float(np.mean((y_true - y_pred) ** 2))  # (실제-예측)^2을 평균 낸 값을 반환한다.

### 코드 4. (a) 왜 `scatter`와 `colorbar`가 필요할까?

점 한 개의 좌표 `X_train[i,0]`가 `x1`, `X_train[i,1]`이 `x2`이다. 이 점의 색깔은 `y_train[i]`로 지정한다. 입력 두 개는 점의 위치로, 정답 하나는 색으로 보이게 되어 3차원 정보를 2차원 그림에 담을 수 있다. `xlim`, `ylim`은 문제에서 지정한 `[-3,3]`을 그대로 사용한다. 그림 파일 `3a_training_scatter.png`가 생성된다.


In [ ]:
# %% 4. [3(a)] 훈련 데이터를 평면에 점으로 그리고 색으로 정답 y를 표현한다.
fig, ax = plt.subplots(figsize=(7, 5.8))  # PNG로 저장할 그래프와 좌표축을 만든다.
scatter = ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="viridis", s=28, alpha=0.85)  # 위치=(x1,x2), 점 색깔=y다.
ax.set_xlabel("x1")  # 가로축에 첫 번째 입력의 이름을 붙인다.
ax.set_ylabel("x2")  # 세로축에 두 번째 입력의 이름을 붙인다.
ax.set_title("3(a) Training points colored by y")  # 과제 소문항과 그래프의 의미를 제목에 적는다.
ax.set_xlim(-3, 3)  # 문제에 주어진 x1 범위를 사용한다.
ax.set_ylim(-3, 3)  # 문제에 주어진 x2 범위를 사용한다.
ax.set_aspect("equal", adjustable="box")  # 두 방향의 거리 척도를 같게 만들어 공간 패턴을 왜곡하지 않는다.
fig.colorbar(scatter, ax=ax, label="observed y")  # 색깔이 어떤 y값을 나타내는지 범례를 붙인다.
fig.tight_layout()  # 글씨와 그래프가 잘리지 않도록 여백을 맞춘다.
fig.savefig(OUTPUT_DIR / "3a_training_scatter.png", dpi=180)  # 3(a)의 제출용 그림 파일을 저장한다.
plt.close(fig)  # 불필요한 그림 메모리를 해제한다.

### 코드 5. (b) RKS는 네 개의 부분을 갖는다

1. **`__init__`**: 설정 D, σ, λ를 저장하고 랜덤 `W`와 `b`를 한 번 뽑는다. `W.shape=(2,h)`, `b.shape=(h,)`. 이후 학습 중 바뀌지 않는다.
2. **`_features`**: `X.shape=(n,2)`를 받고 `X @ W`를 계산하면 `(n,h)`가 된다. `+ b`는 각 행에 같은 위상 벡터를 더하는 **브로드캐스팅**이다. 이어 짝수 열에는 sin, 홀수 열에는 cos를 채워 `phi.shape=(n,D)`를 만든다.
3. **`_design`**: 절편용 1열을 붙여 `A.shape=(n,D+1)`로 만든다.
4. **`fit/predict`**: `AᵀA+λP`로 연립방정식을 풀어 `β`를 저장하고, 새 데이터에서는 `A_new @ β`를 계산한다.

이 모델에서 **학습되는 것은 β만**이고, `W,b`는 랜덤으로 생성만 된다. 이는 MLP와의 결정적인 차이다. 초기화에 `seed`를 고정했기 때문에 D에 따라 특징 수를 늘려도 공통 앞부분의 랜덤 물결이 유지된다.


In [ ]:
# %% 5. [3(b)] RKS 모델: random sin/cos 특성 + Ridge Regression을 직접 구현한다.
class RandomKitchenSinks:  # RKS 모델의 생성, 변환, 학습, 예측을 하나의 클래스로 묶는다.
    def __init__(self, D=80, sigma=1.0, ridge_lambda=0.01, seed=42):  # 학습 전에 정할 하이퍼파라미터를 받는다.
        if D <= 0 or D % 2 != 0:  # sin/cos는 쌍으로 만들므로 D는 양의 짝수여야 한다.
            raise ValueError("D must be a positive even integer")  # 잘못된 D가 입력되면 즉시 알려 준다.
        if sigma <= 0 or ridge_lambda <= 0:  # σ와 λ를 모두 양수로 제한한다.
            raise ValueError("sigma and ridge_lambda must be positive")  # 불가능한 설정을 걸러 낸다.
        self.D = int(D)  # 전체 변환 특성의 개수 D를 저장한다.
        self.h = self.D // 2  # sin/cos 쌍의 개수 h는 D/2이다.
        self.sigma = float(sigma)  # 랜덤 주파수의 스케일을 저장한다.
        self.ridge_lambda = float(ridge_lambda)  # Ridge 규제 강도를 저장한다.
        rng_w = np.random.default_rng(seed)  # 랜덤 주파수 w를 뽑는 독립적인 난수 생성기다.
        rng_b = np.random.default_rng(seed + 1)  # 랜덤 위상 b를 뽑는 별도의 난수 생성기다.
        self.W = rng_w.normal(0.0, 1.0 / self.sigma, size=(self.h, 2)).T  # w_j 두 좌표를 N(0,1/sigma^2)에서 뽑아 (2,h)로 만든다.
        self.b = rng_b.uniform(0.0, 2.0 * np.pi, size=self.h)  # 각 사인파의 시작 위치 b_j를 [0,2π]에서 뽑는다.
        self.coef_ = None  # 아직 학습하지 않았으므로 최종 선형회귀 계수가 없다.

    def _features(self, X):  # 입력 x1,x2를 여러 개의 sin/cos 숫자로 바꾸는 함수다.
        angle = X @ self.W + self.b  # 모든 데이터에 대해 w_j^T x_i + b_j를 계산한다; 결과는 (n,h)다.
        phi = np.empty((len(X), self.D), dtype=np.float64)  # 출력 sin/cos 특징을 담을 (n,D) 빈 행렬을 만든다.
        phi[:, 0::2] = np.sin(angle)  # 짝수 열(0,2,4,...)마다 사인값을 채워 넣는다.
        phi[:, 1::2] = np.cos(angle)  # 홀수 열(1,3,5,...)마다 코사인값을 채워 넣는다.
        return phi  # 한 데이터마다 D개 특징을 가진 행렬을 반환한다.

    def _design(self, X):  # 절편 b0를 포함하는 선형회귀 디자인 행렬을 만든다.
        ones = np.ones((len(X), 1), dtype=np.float64)  # 절편 계수를 곱할 '1' 열을 만든다.
        return np.column_stack((ones, self._features(X)))  # 첫 열 1 + D개 특징을 이어 (n,D+1)을 반환한다.

    def fit(self, X, y):  # 주어진 입력과 정답으로 Ridge의 계수를 학습한다.
        A = self._design(X)  # 모든 데이터에 대한 실제 디자인 행렬 A를 만든다.
        penalty = np.eye(self.D + 1, dtype=np.float64)  # L2 벌점을 주기 위한 (D+1)×(D+1) 단위행렬을 만든다.
        penalty[0, 0] = 0.0  # 첫 계수는 절편이므로 Ridge 벌점을 받지 않게 만든다.
        left = A.T @ A + self.ridge_lambda * penalty  # 정규방정식 왼쪽 A^T A + λP를 계산한다.
        right = A.T @ y  # 정규방정식 오른쪽 A^T y를 계산한다.
        self.coef_ = np.linalg.solve(left, right)  # 역행렬을 직접 계산하지 않고 연립방정식을 풀어 계수를 찾는다.
        return self  # 학습된 객체 자체를 반환해 .fit(...).predict(...)처럼 쓸 수 있게 한다.

    def predict(self, X):  # 학습된 계수로 새 데이터에 대한 y값을 예측한다.
        if self.coef_ is None:  # 아직 fit을 한 적 없는 모델인지 확인한다.
            raise RuntimeError("Call fit() before predict()")  # 학습 전 예측 시 명확한 오류를 낸다.
        return self._design(X) @ self.coef_  # 새로운 디자인 행렬과 계수 벡터를 곱해 예측값을 반환한다.

### 코드 6. 하이퍼파라미터 Sweep은 무슨 실험인가?

`D=20,40,80,160,320`으로 바꾸되 `σ`, `λ`, 훈련 데이터, 랜덤 시드는 고정한다. **한 번에 하나의 조건만 바꿔야** D가 성능에 어떤 영향을 미치는지 해석할 수 있기 때문이다. 각 모델마다 Train MSE와 Validation MSE를 **둘 다** 기록한다. 최선 모델은 `val_mse`가 가장 낮은 D이다. 여기서 Validation y를 사용해 고르는 것은 허용되지만, test y는 존재하지도 않고 사용해서도 안 된다.


In [ ]:
# %% 6. [3(b)] D를 바꿔 보고, Train MSE와 Validation MSE를 모두 기록한다.
rks_rows = []  # 여러 RKS 후보의 평가 기록을 담을 빈 리스트다.
for D in RKS_D_VALUES:  # 미리 정한 D=20,40,80,160,320을 순서대로 시도한다.
    candidate = RandomKitchenSinks(D=D, sigma=RKS_SIGMA, ridge_lambda=RKS_LAMBDA, seed=SEED)  # D 외의 조건을 고정한 후보 모델이다.
    candidate.fit(X_train, y_train)  # 오직 훈련 데이터로 각 후보의 Ridge 계수를 학습한다.
    train_mse = mse(y_train, candidate.predict(X_train))  # 학습 데이터에서의 예측 오차를 기록한다.
    val_mse = mse(y_val, candidate.predict(X_val))  # 보지 않은 검증 데이터의 예측 오차를 기록한다.
    rks_rows.append({"D": D, "train_mse": train_mse, "val_mse": val_mse})  # D와 두 손실을 하나의 기록으로 저장한다.
    print(f"RKS D={D:3d} | train MSE={train_mse:.6f} | val MSE={val_mse:.6f}")  # 각 후보 성능을 즉시 출력한다.
rks_table = pd.DataFrame(rks_rows)  # 위 기록을 CSV로 저장할 수 있는 표로 바꾼다.
rks_table.to_csv(OUTPUT_DIR / "3b_rks_sweep.csv", index=False)  # 각 D의 손실을 실험 데이터 파일로 저장한다.
best_rks_idx = int(rks_table["val_mse"].idxmin())  # 검증 MSE가 최소인 행 번호를 찾는다.
best_D = int(rks_table.loc[best_rks_idx, "D"])  # 그 행에 기록된 최적 D를 선택한다.
print(f"선택한 최적 RKS D: {best_D}")  # 검증 기준으로 선택한 설정을 보여 준다.

### 코드 7. '두 개의 그림'이라는 과제 문구의 의미

교수님은 Sweep에서 `Train MSE vs D`와 `Validation MSE vs D`를 **별도 그림 2개**로 요구했다. `plot_sweep`은 그림 저장 절차를 함수로 만들어 같은 코드를 재사용하도록 한다. `matplotlib.pyplot`의 `plot(x,y)`는 x가 D, y가 MSE라는 뜻이다. 출력 파일 이름에는 과제 번호를 넣었다.


In [ ]:
# %% 7. Hyperparameter sweep 그래프는 train과 validation을 구분해 두 개로 그린다.
def plot_sweep(table, x_name, y_name, title, filename):  # 표의 한 손실 열을 선 그래프로 저장하는 재사용 함수다.
    fig, ax = plt.subplots(figsize=(6.5, 4.5))  # 그림과 좌표축을 생성한다.
    ax.plot(table[x_name], table[y_name], marker="o", linewidth=2)  # 가로축=하이퍼파라미터, 세로축=MSE를 그린다.
    ax.set_xlabel(x_name)  # 가로축에 D 또는 hidden_width를 적는다.
    ax.set_ylabel("Mean squared error (MSE)")  # 세로축에 평가 지표를 적는다.
    ax.set_title(title)  # 무엇을 비교한 그래프인지 제목을 붙인다.
    ax.grid(True, alpha=0.3)  # 값을 눈으로 비교하기 쉽도록 연한 격자를 그린다.
    fig.tight_layout()  # 글씨가 잘리지 않도록 여백을 조정한다.
    fig.savefig(OUTPUT_DIR / filename, dpi=180)  # PNG 이미지로 저장한다.
    plt.close(fig)  # 메모리를 해제한다.

plot_sweep(rks_table, "D", "train_mse", "3(b) RKS training MSE vs D", "3b_rks_training_mse.png")  # 요구된 RKS 학습 손실 그래프다.
plot_sweep(rks_table, "D", "val_mse", "3(b) RKS validation MSE vs D", "3b_rks_validation_mse.png")  # 요구된 RKS 검증 손실 그래프다.

### 코드 8. MLP 클래스는 왜 `nn.Module`을 상속할까?

`nn.Module`은 PyTorch 신경망의 표준 '부모'이다. `super().__init__()`을 호출해야 PyTorch가 `nn.Linear`의 **학습 가능한 행렬과 편향**을 등록하고, `model.parameters()`나 `.to(DEVICE)`가 동작한다. `Linear(2,width)`는 두 입력을 width개 가중합으로, `ReLU`는 음수 부분을 0으로, `Linear(width,1)`는 최종 숫자 1개로 바꾼다. **ReLU는 은닉층이 아니라 활성화 함수**이므로 이 MLP의 은닉층은 정확히 하나다.


In [ ]:
# %% 8. [3(c)] 정확히 은닉층이 하나인 MLP 신경망 클래스를 작성한다.
class OneHiddenMLP(nn.Module):  # PyTorch의 모든 신경망이 상속하는 nn.Module을 부모 클래스로 사용한다.
    def __init__(self, width):  # width는 중간의 은닉 노드를 몇 개 둘지 정하는 숫자다.
        super().__init__()  # PyTorch가 내부 가중치를 정상적으로 추적하도록 부모 생성자를 실행한다.
        self.net = nn.Sequential(  # 층을 적은 순서대로 연결해 하나의 신경망으로 구성한다.
            nn.Linear(2, width),  # (x1,x2)의 두 숫자를 width개의 가중합으로 바꾼다.
            nn.ReLU(),  # 음수는 0, 양수는 그대로 통과시켜 비선형성을 추가한다.
            nn.Linear(width, 1),  # 은닉 표현 width개를 최종 예측값 하나로 바꾼다.
        )  # Sequential 정의를 마친다.

    def forward(self, x):  # 모델에 입력 x가 들어왔을 때 계산할 순서를 정한다.
        return self.net(x).squeeze(1)  # (batch,1) 형태의 출력을 (batch,)로 바꿔 정답 모양과 맞춘다.

### 코드 9. 훈련 함수에서 정말 중요한 실행 순서

`x_train_tensor`, `y_train_tensor`는 NumPy를 Tensor로 변환한다. `TensorDataset`은 x와 y를 같은 행끼리 묶고 `DataLoader`는 64개씩 섞어 제공한다. 모델과 배치에 `.to(DEVICE)`가 없으면 CPU/GPU가 달라져 오류가 발생할 수 있다.

**매 배치마다 5단계**: `zero_grad()`로 옛 기울기 비우기 → `model(batch_x)`로 예측 → `loss_fn`으로 오차 측정 → `backward()`로 미분 → `step()`으로 가중치 업데이트.

**매 Epoch마다 4단계**: `eval()`로 평가 모드 → `no_grad()` 아래 전체 Train/Val MSE 계산 → Validation 최고 기록이면 `deepcopy(model.state_dict())`로 가중치 저장 → 개선이 없으면 카운터를 늘려 45회 연속이면 중단. 마지막의 `load_state_dict(best_state)`는 **최고 Epoch의 가중치**를 복원한다.

참고: 네트워크가 ReLU와 선형층만 써서 train/eval 계산 차이는 크지 않지만, `eval()` 습관은 Dropout/BatchNorm을 사용할 때 꼭 필요하다.


In [ ]:
# %% 9. MLP를 직접 훈련하고 검증 MSE가 가장 좋을 때의 가중치를 보존한다.
def fit_mlp(width):  # 은닉층 너비 하나를 받아 훈련된 모델과 평가기록을 반환한다.
    torch.manual_seed(SEED)  # 후보마다 비교 가능한 초기화를 얻도록 같은 난수 시드를 사용한다.
    model = OneHiddenMLP(width).to(DEVICE)  # 모델을 만들고 CPU 또는 GPU로 이동시킨다.
    x_train_tensor = torch.tensor(X_train, dtype=torch.float32)  # 훈련 입력을 PyTorch 실수 Tensor로 변환한다.
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)  # 훈련 정답을 PyTorch 실수 Tensor로 변환한다.
    dataset = TensorDataset(x_train_tensor, y_train_tensor)  # 같은 행의 입력과 정답을 하나의 샘플로 묶는다.
    loader = DataLoader(dataset, batch_size=MLP_BATCH_SIZE, shuffle=True)  # 미니배치를 만들고 Epoch마다 순서를 섞는다.
    full_train_x = x_train_tensor.to(DEVICE)  # Epoch 마지막에 전체 train MSE를 정확히 계산할 입력을 준비한다.
    full_train_y = y_train_tensor.to(DEVICE)  # Epoch 마지막에 전체 train MSE를 계산할 정답을 준비한다.
    val_x_tensor = torch.tensor(X_val, dtype=torch.float32, device=DEVICE)  # 검증 입력을 같은 계산 장치에 만든다.
    val_y_tensor = torch.tensor(y_val, dtype=torch.float32, device=DEVICE)  # 검증 정답을 같은 계산 장치에 만든다.
    optimizer = torch.optim.Adam(model.parameters(), lr=MLP_LR)  # 모든 학습 가능한 가중치의 업데이트를 Adam이 담당한다.
    loss_fn = nn.MSELoss()  # 훈련에서 (예측-정답)^2 평균을 최소화한다.
    best_val_mse = float("inf")  # 아직 검증 성능이 없으므로 최고 점수를 무한대로 시작한다.
    best_epoch = 0  # 검증 성능이 가장 좋았던 Epoch 번호를 기록한다.
    best_state = copy.deepcopy(model.state_dict())  # 좋은 결과가 나왔을 때 되돌릴 가중치 복사본이다.
    epochs_without_improve = 0  # 검증 손실이 개선되지 않은 연속 Epoch 수다.
    history = []  # 매 Epoch train/val MSE를 담아 학습 과정을 시각화할 기록이다.
    for epoch in range(1, MLP_MAX_EPOCHS + 1):  # 1부터 최대 Epoch 수까지 학습을 반복한다.
        model.train()  # 신경망을 훈련 모드로 전환한다.
        for batch_x, batch_y in loader:  # 훈련 데이터 중 한 미니배치를 가져온다.
            batch_x = batch_x.to(DEVICE)  # 입력 배치를 모델과 같은 장치로 옮긴다.
            batch_y = batch_y.to(DEVICE)  # 정답 배치도 같은 장치로 옮긴다.
            optimizer.zero_grad()  # 지난 배치에서 남은 기울기를 지운다.
            batch_pred = model(batch_x)  # 신경망으로 이번 배치의 y를 예측한다.
            batch_loss = loss_fn(batch_pred, batch_y)  # 이번 배치의 제곱오차 평균을 계산한다.
            batch_loss.backward()  # 연쇄법칙/역전파로 모든 가중치의 기울기를 계산한다.
            optimizer.step()  # 계산된 기울기를 사용해 Adam이 가중치를 갱신한다.
        model.eval()  # 검증 계산을 위해 평가 모드로 전환한다.
        with torch.no_grad():  # 여기서는 평가만 하므로 불필요한 미분 그래프를 만들지 않는다.
            epoch_train = loss_fn(model(full_train_x), full_train_y).item()  # 현 Epoch의 전체 훈련 MSE를 숫자로 얻는다.
            epoch_val = loss_fn(model(val_x_tensor), val_y_tensor).item()  # 현 Epoch의 전체 검증 MSE를 숫자로 얻는다.
        history.append({"epoch": epoch, "train_mse": epoch_train, "val_mse": epoch_val})  # 학습 과정의 한 행을 기록한다.
        if epoch_val < best_val_mse - 1e-8:  # 이전 최고 검증 MSE보다 충분히 개선되었는지 검사한다.
            best_val_mse = epoch_val  # 이번 Epoch을 최고 점수로 등록한다.
            best_epoch = epoch  # 최고 점수가 나온 Epoch 번호를 저장한다.
            best_state = copy.deepcopy(model.state_dict())  # 모델의 최고 가중치 전체를 복사해 보관한다.
            epochs_without_improve = 0  # 성능이 개선됐으므로 대기 카운터를 초기화한다.
        else:  # 검증 MSE가 개선되지 않았다면 실행되는 부분이다.
            epochs_without_improve += 1  # 더 좋아지지 않은 Epoch 수를 하나 늘린다.
        if epochs_without_improve >= MLP_PATIENCE:  # 연속으로 너무 오래 좋아지지 않았는지 확인한다.
            break  # 과적합과 불필요한 계산을 피하기 위해 훈련 반복을 종료한다.
    model.load_state_dict(best_state)  # 마지막 Epoch이 아닌 '검증 MSE가 가장 작았던' 가중치로 복원한다.
    model.eval()  # 최종 예측 시에는 평가 모드로 둔다.
    history_table = pd.DataFrame(history)  # Epoch별 기록을 데이터프레임으로 변환한다.
    final_train_mse = mse(y_train, predict_mlp(model, X_train))  # 복원한 최적 가중치의 train MSE를 계산한다.
    final_val_mse = mse(y_val, predict_mlp(model, X_val))  # 복원한 최적 가중치의 val MSE를 계산한다.
    return model, best_epoch, final_train_mse, final_val_mse, history_table  # 학습된 모델과 평가 결과를 모두 돌려준다.

### 코드 10. `predict_mlp`에서는 왜 `no_grad`와 `.cpu().numpy()`를 쓸까?

예측할 때는 가중치를 고칠 필요가 없다. 그래서 `torch.no_grad()`는 미분 그래프 계산과 메모리 소모를 막는다. 모델이 GPU에 있어도 CSV와 NumPy 지표 계산은 보통 CPU 배열을 받으므로 `.detach().cpu().numpy()`로 변환한다. `chunk_size=4096`은 160×160=25,600개의 격자도 메모리 부담 없이 **원래 순서대로** 예측하게 해 준다.


In [ ]:
# %% 10. PyTorch 모델로 예측하고 NumPy 배열로 되돌리는 함수를 만든다.
def predict_mlp(model, X, chunk_size=4096):  # 데이터가 아주 많을 때도 나눠 예측할 수 있는 함수다.
    model.eval()  # Dropout 등 학습/평가 동작이 다를 수 있는 층을 평가 모드로 둔다.
    pieces = []  # 여러 예측 배치의 결과를 쌓을 빈 리스트다.
    with torch.no_grad():  # 예측 시에는 역전파가 필요 없으므로 미분 기록을 끈다.
        for start in range(0, len(X), chunk_size):  # chunk_size만큼 입력 행을 나눠 처리한다.
            batch = X[start:start + chunk_size]  # 입력 배열에서 현재 예측 배치만 꺼낸다.
            tensor = torch.tensor(batch, dtype=torch.float32, device=DEVICE)  # 배치를 신경망과 같은 장치의 Tensor로 만든다.
            predicted = model(tensor).detach().cpu().numpy()  # 예측 Tensor를 CPU NumPy 배열로 바꾼다.
            pieces.append(predicted)  # 이번 배치 결과를 순서대로 보관한다.
    return np.concatenate(pieces) if pieces else np.empty(0, dtype=np.float32)  # 모든 예측을 원래 행 순서대로 합친다.

### 코드 11. MLP의 하이퍼파라미터는 무엇을 바꾸는가?

오직 은닉 노드 수 `H`를 `16,32,64,128`로 바꾼다. 다른 조건(Adam 학습률, 배치 크기, 최대 Epoch, 조기종료 기준, 시드)은 고정한다. 각 후보의 최적 Epoch에서 계산한 Train/Validation MSE를 기록한다. Validation 값이 가장 작은 `H`를 선택한다. Train MSE가 더 낮은 모델이 다른 데이터를 더 잘 예측한다고 보장하지 않으니 **선택 기준은 반드시 Validation**이다.


In [ ]:
# %% 11. [3(c)] 은닉층 width를 바꾸고 학습/검증 MSE를 표로 기록한다.
mlp_rows = []  # width별 평가 결과를 저장할 리스트를 만든다.
for width in MLP_WIDTH_VALUES:  # 16,32,64,128개 노드인 모델을 차례대로 비교한다.
    candidate, epoch, train_mse, val_mse, history_table = fit_mlp(width)  # 하나의 모델을 학습하고 조기종료 결과를 받는다.
    mlp_rows.append({"hidden_width": width, "best_epoch": epoch, "train_mse": train_mse, "val_mse": val_mse})  # 선택 기준을 기록한다.
    history_table.to_csv(OUTPUT_DIR / f"3c_mlp_history_width_{width}.csv", index=False)  # 해당 width의 Epoch별 학습곡선을 보존한다.
    print(f"MLP width={width:3d} | best epoch={epoch:3d} | train MSE={train_mse:.6f} | val MSE={val_mse:.6f}")  # 후보 성능을 출력한다.
mlp_table = pd.DataFrame(mlp_rows)  # 결과 리스트를 표로 만든다.
mlp_table.to_csv(OUTPUT_DIR / "3c_mlp_sweep.csv", index=False)  # 전체 width의 결과를 CSV에 저장한다.
best_mlp_idx = int(mlp_table["val_mse"].idxmin())  # 검증 손실이 가장 낮은 행을 찾는다.
best_width = int(mlp_table.loc[best_mlp_idx, "hidden_width"])  # 해당 행의 최적 width를 선택한다.
print(f"선택한 최적 MLP hidden width: {best_width}")  # MLP의 최적 설정을 출력한다.
plot_sweep(mlp_table, "hidden_width", "train_mse", "3(c) MLP training MSE vs width", "3c_mlp_training_mse.png")  # width별 Train MSE 그래프를 저장한다.
plot_sweep(mlp_table, "hidden_width", "val_mse", "3(c) MLP validation MSE vs width", "3c_mlp_validation_mse.png")  # width별 Validation MSE 그래프를 저장한다.

### 코드 12. '최적 하이퍼파라미터로 다시 fit'은 무슨 뜻인가?

과제 3(d)는 (b)와 (c)에서 골랐던 설정으로 최종 두 모델을 **새롭게 학습하여 비교**하라고 한다. 여기서는 최적 `D`, 최적 `H`를 사용해 처음부터 다시 학습한다. Validation은 Early Stopping/선택용으로 계속 유지하므로 **Train+Validation을 합쳐 학습하지 않는다.** 합치면 같은 Validation MSE를 독립 비교 수치처럼 보고할 수 없기 때문이다.


In [ ]:
# %% 12. [3(d)] 선택한 설정으로 두 모델을 다시 학습하고 검증 손실을 비교한다.
final_rks = RandomKitchenSinks(D=best_D, sigma=RKS_SIGMA, ridge_lambda=RKS_LAMBDA, seed=SEED)  # 검증으로 선택한 D를 사용해 RKS를 새로 만든다.
final_rks.fit(X_train, y_train)  # 오직 훈련 데이터만으로 RKS의 최종 계수를 다시 계산한다.
final_mlp, final_epoch, mlp_train_mse, mlp_val_mse, final_history = fit_mlp(best_width)  # 선택한 width의 MLP를 초기화부터 다시 학습한다.
rks_train_mse = mse(y_train, final_rks.predict(X_train))  # 최종 RKS의 훈련 MSE를 계산한다.
rks_val_mse = mse(y_val, final_rks.predict(X_val))  # 최종 RKS의 검증 MSE를 계산한다.
comparison = pd.DataFrame([  # 두 모델의 성능을 하나의 표로 만든다.
    {"model": "RKS", "selected_hyperparameter": f"D={best_D}", "train_mse": rks_train_mse, "val_mse": rks_val_mse},  # RKS 결과 한 행이다.
    {"model": "MLP", "selected_hyperparameter": f"hidden_width={best_width}", "train_mse": mlp_train_mse, "val_mse": mlp_val_mse},  # MLP 결과 한 행이다.
])  # DataFrame 생성이 끝난다.
comparison.to_csv(OUTPUT_DIR / "3d_model_comparison.csv", index=False)  # 제출 보고서의 수치표를 CSV로 저장한다.
final_history.to_csv(OUTPUT_DIR / "3d_final_mlp_learning_curve.csv", index=False)  # 최종 MLP 학습곡선도 저장한다.
print("\n[3(d)] 최종 검증 성능\n" + comparison.to_string(index=False))  # 비교 표를 터미널에 출력한다.

### 코드 13. 정답 없는 `test.csv`로는 무엇을 할 수 있을까?

`final_rks.predict(X_test)`와 `predict_mlp(final_mlp,X_test)`는 모두 `(200,)` 길이의 숫자 배열을 만든다. `np.savetxt(..., fmt="%.10f")`는 **헤더 없이 행마다 숫자 한 개**만 기록한다. `X_test`를 섞거나 정렬하지 않으므로 교수님이 요구한 **원본 test.csv의 행 순서**가 보존된다. `test.csv`에는 실제 y가 없기 때문에 절대로 Test MSE를 만들거나 가짜 수치를 적지 않는다.


In [ ]:
# %% 13. Test는 정답이 없으므로 예측값만 만들고 행 순서 그대로 CSV로 내보낸다.
rks_test_pred = final_rks.predict(X_test)  # 최종 RKS로 test.csv의 각 행을 예측한다.
mlp_test_pred = predict_mlp(final_mlp, X_test)  # 최종 MLP로 같은 test.csv 행들을 예측한다.
assert len(rks_test_pred) == len(X_test)  # RKS 예측 행 수가 테스트 데이터 행 수와 같은지 검사한다.
assert len(mlp_test_pred) == len(X_test)  # MLP 예측 행 수도 원본과 같은지 검사한다.
assert np.isfinite(rks_test_pred).all() and np.isfinite(mlp_test_pred).all()  # NaN 또는 무한대 예측값이 있는지 검사한다.
np.savetxt("rks_test_predictions.csv", rks_test_pred, fmt="%.10f", delimiter=",")  # 헤더 없이 행마다 예측 숫자 하나인 RKS 제출 CSV를 저장한다.
np.savetxt("mlp_test_predictions.csv", mlp_test_pred, fmt="%.10f", delimiter=",")  # 헤더 없이 행마다 예측 숫자 하나인 MLP 제출 CSV를 저장한다.
print("테스트 예측 저장 완료: rks_test_predictions.csv / mlp_test_predictions.csv")  # 실제 파일명을 알려 준다.

### 코드 14. 예측 '표면(surface)'은 어떻게 그림이 될까?

모델은 `(x1,x2)` 한 점의 높이만 알려 준다. 지도처럼 보이려면 -3~3까지 가로/세로 160개 좌표를 만들고, 모든 조합 **160×160=25,600개 점**을 예측해야 한다. `meshgrid`는 격자, `ravel`은 2D 격자를 1D 순서로 펼치기, `column_stack`은 `[[x1,x2],...]` 형태 만들기, `reshape`는 예측값을 다시 격자 모양으로 되돌리는 것이다.


In [ ]:
# %% 14. [3(d)] [-3,3]×[-3,3] 전체를 촘촘한 격자로 만들어 두 모델의 곡면을 그린다.
axis_values = np.linspace(-3.0, 3.0, 160)  # -3부터 3까지 같은 간격으로 160개의 좌표를 만든다.
mesh_x1, mesh_x2 = np.meshgrid(axis_values, axis_values)  # 모든 (x1,x2) 격자 조합을 만든다.
grid_inputs = np.column_stack((mesh_x1.ravel(), mesh_x2.ravel()))  # (160^2,2) 모양의 예측 입력 행렬로 바꾼다.
rks_grid_pred = final_rks.predict(grid_inputs).reshape(mesh_x1.shape)  # RKS 예측을 160×160 격자 모양으로 재배열한다.
mlp_grid_pred = predict_mlp(final_mlp, grid_inputs).reshape(mesh_x1.shape)  # MLP 예측을 동일한 격자로 재배열한다.
color_low = float(min(y_train.min(), rks_grid_pred.min(), mlp_grid_pred.min()))  # 세 그림에서 색깔 기준의 최솟값을 통일한다.
color_high = float(max(y_train.max(), rks_grid_pred.max(), mlp_grid_pred.max()))  # 세 그림에서 색깔 기준의 최댓값을 통일한다.

### 코드 15. 두 모델의 색깔을 같은 기준으로 비교해야 한다

RKS 그래프와 MLP 그래프의 색 눈금이 다르면 '밝기가 더 밝다'는 착시가 생긴다. 그래서 세 그림의 공통 최솟값/최댓값을 계산하여 `vmin`, `vmax`에 같은 숫자를 사용한다. `contourf`는 값이 비슷한 위치끼리 색으로 채운다. 결과는 `3d_rks_surface.png`와 `3d_mlp_surface.png`이다.


In [ ]:
# %% 15. 공통 색 범위를 사용해 각 예측 곡면의 독립된 그림을 저장한다.
def save_surface(Z, title, filename):  # 예측 격자 Z와 파일명을 받아 2D 등고 색지도를 저장한다.
    fig, ax = plt.subplots(figsize=(7, 5.8))  # 그림과 좌표축을 만든다.
    contour = ax.contourf(mesh_x1, mesh_x2, Z, levels=35, cmap="viridis", vmin=color_low, vmax=color_high)  # 격자에서 예측값의 공간 분포를 색깔로 표현한다.
    ax.set_xlabel("x1")  # x1 좌표축 이름을 표시한다.
    ax.set_ylabel("x2")  # x2 좌표축 이름을 표시한다.
    ax.set_title(title)  # RKS 또는 MLP 표면인지 제목을 표시한다.
    ax.set_xlim(-3, 3)  # 문제의 입력 범위와 동일하게 가로축 범위를 정한다.
    ax.set_ylim(-3, 3)  # 문제의 입력 범위와 동일하게 세로축 범위를 정한다.
    ax.set_aspect("equal", adjustable="box")  # 가로/세로 축의 척도를 같게 한다.
    fig.colorbar(contour, ax=ax, label="predicted y")  # 색상에 대응하는 예측 y 크기를 표시한다.
    fig.tight_layout()  # 라벨 잘림을 방지한다.
    fig.savefig(OUTPUT_DIR / filename, dpi=180)  # 해당 예측 표면을 PNG로 저장한다.
    plt.close(fig)  # 그림 객체를 닫는다.

save_surface(rks_grid_pred, f"3(d) RKS fitted surface, D={best_D}", "3d_rks_surface.png")  # RKS 최종 예측 공간 그림을 저장한다.
save_surface(mlp_grid_pred, f"3(d) MLP fitted surface, width={best_width}", "3d_mlp_surface.png")  # MLP 최종 예측 공간 그림을 저장한다.

### 코드 16. 진짜 패턴을 학습했는지 눈으로 판단하자

(a)에서 그린 훈련 y의 산점도와 두 모델의 표면을 같은 크기와 색 눈금으로 배치한다. 원래 데이터에서 높은 영역이 어딘지, 낮은 영역이 어딘지 먼저 본다. 그 다음 두 모델이 **저점·고점 위치와 부드러운 변화**를 얼마나 잘 보존하는지 비교한다. 검증 MSE라는 숫자와 이 그림을 **함께** 사용해야 좋은 해석이 된다.


In [ ]:
# %% 16. 훈련 자료의 패턴과 RKS/MLP의 예측을 같은 눈금으로 나란히 비교한다.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharex=True, sharey=True)  # 왼쪽부터 실제 데이터, RKS, MLP 세 패널을 만든다.
axes[0].scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="viridis", vmin=color_low, vmax=color_high, s=12)  # 훈련 데이터의 실제 색을 표시한다.
axes[0].set_title("Observed training y")  # 첫 패널의 의미를 적는다.
axes[1].contourf(mesh_x1, mesh_x2, rks_grid_pred, levels=35, cmap="viridis", vmin=color_low, vmax=color_high)  # RKS의 예상 공간 패턴을 표시한다.
axes[1].set_title(f"RKS, D={best_D}")  # 두 번째 패널의 모델 및 설정을 적는다.
axes[2].contourf(mesh_x1, mesh_x2, mlp_grid_pred, levels=35, cmap="viridis", vmin=color_low, vmax=color_high)  # MLP의 예상 공간 패턴을 표시한다.
axes[2].set_title(f"MLP, width={best_width}")  # 세 번째 패널의 모델 및 설정을 적는다.
for ax in axes:  # 세 좌표축에 동일한 표시 범위를 적용한다.
    ax.set_xlabel("x1")  # 가로축 이름을 통일한다.
    ax.set_ylabel("x2")  # 세로축 이름을 통일한다.
    ax.set_xlim(-3, 3)  # 같은 x1 범위를 사용한다.
    ax.set_ylim(-3, 3)  # 같은 x2 범위를 사용한다.
    ax.set_aspect("equal", adjustable="box")  # 세 패널의 공간 크기 비율을 같게 한다.
fig.tight_layout()  # 세 개의 그림 간격을 알맞게 배치한다.
fig.savefig(OUTPUT_DIR / "3d_three_panel_comparison.png", dpi=180)  # (d)에서 가장 핵심인 비교 그림을 저장한다.
plt.close(fig)  # 불필요한 그래프 메모리를 정리한다.

### 코드 17. '훈련 오차는 줄어드는데 검증 오차가 커진다'의 의미

학습이 오래 진행되면 MLP는 훈련점을 더 잘 맞히더라도 보지 않은 검증점에서는 더 틀릴 수 있다. 이것이 **과적합(overfitting)**의 한 단서다. 두 곡선을 동시에 그려 가장 검증 MSE가 좋았던 `best_epoch`에 점선을 그린다. 모델은 이미 이 Epoch의 가중치를 복원한 상태다.


In [ ]:
# %% 17. 최종 모델 MLP가 어떻게 학습되었는지 검증/훈련 손실을 같은 그림에 기록한다.
fig, ax = plt.subplots(figsize=(7, 4.5))  # Early Stopping 학습곡선 그래프를 만든다.
ax.plot(final_history["epoch"], final_history["train_mse"], label="Train MSE")  # Epoch별 훈련 오차를 그린다.
ax.plot(final_history["epoch"], final_history["val_mse"], label="Validation MSE")  # Epoch별 검증 오차를 그린다.
ax.axvline(final_epoch, linestyle="--", label=f"Best epoch={final_epoch}")  # 최적 검증 오차를 얻은 Epoch을 표시한다.
ax.set_xlabel("Epoch")  # 가로축의 단위를 명시한다.
ax.set_ylabel("MSE")  # 세로축의 손실 지표를 명시한다.
ax.set_title("3(d) Final MLP learning curves")  # 그래프의 제목을 붙인다.
ax.legend()  # 선들의 이름을 표시한다.
ax.grid(True, alpha=0.3)  # 약한 격자를 넣어 숫자 추세를 보기 쉽게 한다.
fig.tight_layout()  # 그림이 잘리지 않도록 정리한다.
fig.savefig(OUTPUT_DIR / "3d_mlp_learning_curve.png", dpi=180)  # 학습과 검증이 달라지는 과정을 PNG로 저장한다.
plt.close(fig)  # 그림 객체를 닫는다.

### 코드 18. 제출 문장을 숫자로 자동 채우기

RKS와 MLP의 `train_mse`, `val_mse`와 최적 하이퍼파라미터를 텍스트에 채워 `3_submittable_english_draft.txt`에 저장한다. 검증값을 근거로 어느 모델이 더 좋았는지 쓰고, 세 패널 그림에서 본 두 곡면의 차이를 해석으로 덧붙인다. 테스트에는 정답이 없으므로 테스트 성능 수치는 쓰지 않는다.


In [ ]:
# %% 18. 영문 제출 초안을 실제 실행 수치로 자동 채운다 (해석은 그래프를 보고 검토할 것).
winner = "RKS" if rks_val_mse < mlp_val_mse else "MLP"  # 더 작은 검증 MSE를 낸 모델명을 정한다.
english_summary = f"""Problem 3: Random Kitchen Sinks versus One-Hidden-Layer MLP

(a) Data visualization
We plotted the 600 labeled training examples as a 2D scatter plot, with x1 and x2 on the axes and observed y represented by color. See 3a_training_scatter.png.

(b) Random Kitchen Sinks with ridge regression
We transformed each two-dimensional input into D sinusoidal features: sin(w_j^T x+b_j) and cos(w_j^T x+b_j), where w_j ~ N(0,I_2/sigma^2), b_j ~ Uniform(0,2pi), h=D/2, sigma={RKS_SIGMA}, and the random seed was fixed at {SEED}. We fit an L2-regularized least-squares model with lambda={RKS_LAMBDA}, excluding the intercept from the penalty. We swept D over {RKS_D_VALUES} and chose D={best_D} by validation MSE. The refitted model achieved train MSE={rks_train_mse:.6f} and validation MSE={rks_val_mse:.6f}. See 3b_rks_training_mse.png and 3b_rks_validation_mse.png.

(c) One-hidden-layer MLP
We trained a neural network with two input units, one fully connected ReLU hidden layer, and one linear output. We swept hidden layer width over {MLP_WIDTH_VALUES}, fixing Adam learning rate at {MLP_LR}, batch size at {MLP_BATCH_SIZE}, and early-stopping patience at {MLP_PATIENCE}. Using the validation MSE criterion, we selected width={best_width}. See 3c_mlp_training_mse.png and 3c_mlp_validation_mse.png.

(d) Comparison and visualization
We refitted each selected model on the training split and evaluated both against the labeled validation split. RKS validation MSE={rks_val_mse:.6f}; MLP validation MSE={mlp_val_mse:.6f}. The lower validation MSE was achieved by {winner}. Both predictions on the unlabeled 200-row test split were exported, preserving the original row order, to rks_test_predictions.csv and mlp_test_predictions.csv (headerless, one scalar per row). See 3d_three_panel_comparison.png, 3d_rks_surface.png, and 3d_mlp_surface.png.

Interpretation: RKS uses a fixed, randomly constructed smooth sinusoidal basis and fits only the linear output weights with ridge regression; the MLP learns both its hidden features and its output weights through backpropagation. In the three-panel figure both surfaces recover the trough near the left-center and the higher values toward the top and bottom; the RKS surface is smoother, while the ReLU MLP surface is slightly more piecewise-linear. No test MSE is reported because test.csv has no ground-truth y.
"""  # test에는 정답 y가 없으므로 위 문자열에는 검증 결과만 숫자로 적는다.
(OUTPUT_DIR / "3_submittable_english_draft.txt").write_text(english_summary, encoding="utf-8")  # 수치가 채워진 영문 초안을 텍스트 파일로 저장한다.
print("\n3번 전체 완료: problem3_outputs 폴더, 두 test 예측 CSV, 영문 제출 초안을 확인하세요.")  # 모든 작업이 완료되었음을 알려 준다.

### 모든 코드를 실행했다면 결과 그림을 노트북 안에서도 확인하자

다음 셀은 `problem3_outputs/`의 PNG를 불러와 화면에 표시한다. (화면 표시 도구는 노트북에만 필요하므로 `.py`에는 없다.)

In [ ]:
from IPython.display import display, Image  # 노트북 셀에 PNG 결과를 직접 표시하는 IPython 도구다.
from pathlib import Path  # PNG 파일이 있는 폴더 경로를 만들어 준다.
for image_name in ["3a_training_scatter.png", "3b_rks_training_mse.png", "3b_rks_validation_mse.png", "3c_mlp_training_mse.png", "3c_mlp_validation_mse.png", "3d_three_panel_comparison.png"]:  # 중요한 여섯 그림을 순서대로 본다.
    print(image_name)  # 지금 확인할 그림 파일 이름을 표시한다.
    display(Image(filename=str(Path("problem3_outputs") / image_name), width=820))  # 해당 PNG를 화면에 보여 준다.


---

# Part E. 실제 실험 결과를 해석하는 방법

## 13. 교수님 데이터에 대해 실행한 숫자

**RKS: D=20, σ=1, λ=0.01**

| D | Train MSE | Validation MSE |
|---:|---:|---:|
| 20 | 0.035776 | **0.036124** |
| 40 | 0.032861 | 0.036553 |
| 80 | 0.031879 | 0.036887 |
| 160 | 0.030390 | 0.038196 |
| 320 | 0.029573 | 0.041269 |

**MLP: LR=0.002, batch=64, 최대 600 Epoch, patience=45**

| Hidden width | Best Epoch | Train MSE | Validation MSE |
|---:|---:|---:|---:|
| 16 | 524 | 0.037643 | 0.042134 |
| 32 | 279 | 0.036007 | 0.042949 |
| 64 | 407 | 0.030971 | 0.039303 |
| 128 | 204 | 0.031993 | **0.038674** |

**최종 비교**: RKS validation MSE **0.036124** vs MLP **0.038674**. 이 설정과 데이터에서는 RKS가 약간 더 낫다. 단, 이 차이가 모든 랜덤 시드나 다른 설정에서도 유지된다고 말하려면 반복 실험이 필요하다.

### 왜 D가 늘어나면 훈련은 좋아지고 검증은 나빠질까?

RKS의 D가 20→320으로 커지면 **표현할 수 있는 곡선이 많아져** Train MSE가 0.035776→0.029573으로 떨어졌다. 반면 Validation MSE는 0.036124→0.041269로 증가했다. 이는 많은 특징이 훈련 데이터의 우연한 변동에도 맞아들어갈 수 있다는 **일반화 격차(generalization gap)**의 증거다. 여기서는 `λ`를 고정했기 때문에 D가 달라지면서 규제 강도의 *상대적 효과*도 바뀔 수 있다.

### 왜 MLP가 훈련 데이터에서는 더 잘 맞히는데 검증 데이터에서는 뒤질까?

최적 MLP의 Train MSE `0.031993`는 RKS의 `0.035776`보다 낮다. 하지만 검증에서는 MLP가 더 나쁘다. 즉 **훈련 데이터를 더 잘 맞힌 것과 새로운 데이터를 잘 맞히는 것은 다르다.** MLP는 은닉 특징까지 학습할 수 있어 유연하지만 그 유연성이 일부에서는 더 큰 train–validation 격차로 나타날 수 있다. 단정적으로 '항상 MLP가 과적합된다'고 해석하지는 않는다.

### 그림으로 해석해야 할 세부사항

세 패널 그림에서 실제 관측은 왼쪽 중앙 근처에 낮은 값, 상단·하단에 높은 값을 보인다. RKS와 MLP는 이 큰 구조를 모두 재현했다. RKS 표면은 사인/코사인 조합답게 전반적으로 매끄럽고, ReLU MLP는 일부 영역에서 더 각진 변화가 드러난다. 이번 검증 MSE 차이는 작으므로 **시각적으로 비슷해 보여도** 수치 기준의 우열이 갈릴 수 있다.

## 14. 교수님 요구조건 체크

- [x] 3(a): 훈련 산점도 `x1, x2`와 색으로 표현한 `y`
- [x] 3(b): Sin/Cos RKS 직접 구현 + Ridge + D sweep + 훈련/검증 MSE **각각 별도 그림**
- [x] 3(c): 정확히 은닉층 1개인 PyTorch MLP + width sweep + 훈련/검증 MSE **각각 별도 그림**
- [x] 3(d): 최적 설정 재학습, Validation MSE 보고, 두 예측곡면 그림, 원래 데이터와 비교
- [x] `rks_test_predictions.csv`, `mlp_test_predictions.csv`: 테스트 200행과 같은 순서로 **숫자 하나씩**
- [x] 전체 `.py` 코드, 단계별 `.ipynb`, 영어 제출 초안
- [x] Test는 정답이 없으므로 테스트 MSE는 보고하지 않음

**학문적 해석 주의**: 검증 성능 차이가 통계적으로 유의한지 이 단일 split만으로 증명되지는 않는다. 예측 파일의 값은 실제 제공된 `test.csv`에서 계산했지만, 정답이 없어 최종 채점 성적을 알 수 없다.

## 15. 파일의 역할

| 파일 | 어디에 사용하는가 |
|---|---|
| `code/problem3_rks_mlp.py` | 실제 제출용 전체 구현; 코드 줄마다 한국어 주석 |
| `notebooks/problem3_study.ipynb` | 지금 읽는 초등학생→전문가 학습용 노트북 (영어판: `notebooks_en/`) |
| `problem3_outputs/` | (a)~(d) 그림, sweep 표, 영문 초안 (`run_all.py`로 실행하면 `outputs/problem3/problem3_outputs/`) |
| `submissions/rks_test_predictions.csv`, `submissions/mlp_test_predictions.csv` | 3(d) 요구 제출물 |

### 제출 전에 다시 확인할 점

`homework_2` 폴더에서 `python run_all.py --problem 3`로 **실제 실행에 성공하는지** 확인한다. 제출할 때에는 그래프와 숫자가 보고서의 해석과 일치하는지 다시 확인한다.

